# Observational comparison: NSB Standard, experimental Oxygen and nsb2

This notebook compares the **same registered Patat (2008) FORS1 observations**
in ground-level photon surface radiance integrated over **365–650 nm**.
The goal is to expose improvements and degradations, not establish NSB superiority.

**Oxygen is experimental. These observations influenced its development.**
The historical evaluation partition is not an untouched holdout. Nothing here
promotes Oxygen to Automatic or claims site calibration or general optical accuracy.

Start with the [setup and protocol](README.md). Prepare the locked isolated kernel
before Run All. Installation and exact Git checkout are explicit setup operations;
analysis cells never install packages or execute downloaded code.

## 1. Configure the experiment

Modes: `offline` exercises synthetic fixtures without network/model imports;
`historical` displays rounded published development evidence;
`current` evaluates real observations against the separately verified runtime.
The source commit is an explicit identity, not a claim that 0.2.0 is published.

For real data, set `mode='current'` and the receipt generated by the README's
Maturin command. Keep cache outside Git. `offline_data=True` requires complete
verified caches. A missing measured solar input marks that scenario unavailable.

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown, Image
from nsb_observational.experiment import Config, prepare, evaluate, analyze, export
from nsb_observational.report import HISTORICAL, LIMITATIONS, verify

config = Config(
    mode=os.environ.get('NSB_VALIDATION_MODE', 'offline'),
    cache=os.environ.get('NSB_VALIDATION_CACHE', str(Path.home()/'.cache'/'nsb-observational')),
    output=os.environ.get('NSB_VALIDATION_OUTPUT', 'validation-results'),
    receipt=os.environ.get('NSB_VALIDATION_RECEIPT', ''),
    nsb_commit=os.environ.get('NSB_VALIDATION_COMMIT', 'f5e1d22689ed417c0dcf9df72da54365c30f5bd5'),
    runtime_mode='source',   # or 'published', with explicit nsb_version
    nsb_version=None,
    offline_data=os.environ.get('NSB_VALIDATION_OFFLINE_DATA', '0') == '1',
    solar_file=os.environ.get('NSB_VALIDATION_SOLAR') or None,
    repetitions=2000,
    seed=182,
)
print(config)

## 2. Historical evidence stays separate

The original benchmark, Oxygen numerical development, and merged implementation
have distinct executable identities. These rounded scores are context, not results
of this kernel. Matched-F10.7 superiority over nsb2 was **not demonstrated**;
the paired interval includes zero. The baseline outperformed both in relative MAE.

In [ ]:
display(Markdown('Historical relative MAE (development evidence):'))
for name, value in HISTORICAL['relative_mae'].items():
    print(f'{name:32s} {value:.5f}')
print(HISTORICAL['matched_superiority'])
print('This run mode:', config.mode)

## 3. Verify runtime, acquire observations, preserve the cohort

The full SHA-256-verified Patat TAR is safely extracted and its FITS inventory
checked on every reuse. Frozen metadata cuts admit 103 spectra/22 nights;
94 spectra/17 nights retain the historical evaluation label. Five separate nights
fit only the baseline. All identities and exclusions are exported.

Energy-to-photon conversion uses $I_{ph}(\lambda)=I_E(\lambda)\lambda/(hc)$,
explicit Angstrom/cm, square-arcsecond/steradian and second/nanosecond factors.
The exact piecewise-linear energy integral is independent of NSB ground truth.
No unmatched magnitudes or detector counts are compared.

In [ ]:
state = prepare(config)
print('Observations:', len(state['rows']))
print('Scenario availability:', state['availability'])
if state['provenance'].get('selection'):
    print('Frozen cohort identity:', state['provenance']['selection']['identity_sha256'])
if state['provenance'].get('solar'):
    print('Outside PALACE 67–166 sfu:', state['provenance']['solar']['outside_palace_training'])

## 4. Evaluate three explicit configurations

NSB Standard selects `PARANAL_PALACE_V1`; Oxygen selects the actual
`PARANAL_PALACE_OXYGEN_V1` implementation. Restricted-band Zodiacal, Airglow
and physically zero Moon are evaluated explicitly. Starlight retains the historical
**research-only** solar-photon redistribution of the integrated production map.
nsb2 uses its pinned public source-level radiance factories, including the faint
Gaia population and combined SkyCalc Airglow; no telescope response is applied.
Component populations and atmospheric operators differ.

Native inputs and complete centered 27-day independently measured F10.7 inputs
are separate scenarios. No residual fitting or invented solar values is allowed.

In [ ]:
state = evaluate(state)
print('Rejected predictions:', len(state['failures']))

## 5. Paired statistics on registered evaluation identities

Every successful scenario requires every registered observation for every model.
Missing/nonfinite predictions invalidate the scenario rather than shrinking its
cohort. Report bias, MAE, RMSE, median AE, mean/median relative AE and counts.

The baseline is the calibration-night median. Each spectrum receives equal
weight (historically “exposure-weighted”, not duration-weighted). Bootstrap whole
nights 2,000 times with seed 182, using the same draws for all paired differences.
95% percentile intervals describe sampling variability conditional on the baseline,
not shared flux-calibration uncertainty. Improvement does not imply admission.

In [ ]:
state = analyze(state)
for scenario, result in state['metrics'].items():
    display(Markdown(f'### {scenario}'))
    table = '| Model | N | Nights | Relative MAE | 95% CI |\n|---|---:|---:|---:|---|\n'
    for name, score in result['models'].items():
        table += f"| {name} | {score['n']} | {score['nights']} | {score['mean_absolute_relative_error']:.5f} | {score['relative_mae_ci95']} |\n"
    display(Markdown(table))
    display(result['paired'])

## 6. Export a new immutable report and inspect the figures

Eight figure families show both favorable and unfavorable cases. Spectral regions
compare **observed total radiance on air wavelengths** with **NSB Airglow in
1 nm vacuum bins**. Model delta lines are not instrument-convolved sky spectra.
PNG and editable SVG outputs accompany Markdown/HTML and machine-readable data.
A detached checksum covers the complete non-self-referential manifest.

In [ ]:
output = export(state)
manifest = verify(output)
print('Report:', (output/'report.html').resolve())
print('Verified manifest:', (output/'manifest.sha256').read_text().strip())
for figure in sorted((output/'figures').glob('*.png')):
    display(Markdown(f'**{figure.stem}**'))
    display(Image(filename=str(figure)))

## 7. Interpretation and reproducibility

Inspect all paired confidence intervals, degradations, solar extrapolation and
night concentration before interpreting aggregate means. Results from a new
commit receive a new manifest, even if aggregate numbers resemble historical
scores. Historical artifacts remain unchanged. Optional per-observation historical
checks and their tolerances are documented in [README.md](README.md).

The report records code/file hashes, runtime receipt, native extension, wheel,
Cargo.lock, Rust/Maturin, Python/provider/dependency versions, dataset, solar and
source-asset identities, exclusions, partition, settings, metrics and output hashes.
Run All again produces a separate directory. Archive it with its verified inputs.

In [ ]:
display(Markdown(LIMITATIONS))
if config.mode == 'offline':
    display(Markdown('**All displayed computed values and figures are SYNTHETIC fixtures. No empirical validation was performed.**'))
elif config.mode == 'historical':
    display(Markdown('**Historical display only; no historical binary was replayed.**'))
else:
    display(Markdown('**New development experiment; independent scientific accuracy is not admitted.**'))
if config.mode == 'current':
    import json
    admission = json.loads((output/'admission.json').read_text())
    if not admission['experiment_complete']:
        raise RuntimeError('Incomplete experiment; inspect the exported admission report and scenario availability')
